In [1]:
import pandas as pd
df=pd.read_csv("../datasets/retail-orders-raw.csv")
df.head()

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [2]:
#checking kpi critical fields
print(df[['quantity','unit_price','discount_pct','payment_status']])

   quantity  unit_price  discount_pct payment_status
0         2         799          10.0           Paid
1         1        1499           0.0           paid
2         1        1499           NaN        Pending
3         3         799           5.0           Paid
4         3         799           5.0           Paid
5         1         999           0.0         Failed
6        -1         799          10.0           Paid
7         2         999         105.0           Paid
8       two        1499           0.0        Pending
9         1         799           0.0           Paid
10        2        1499          15.0       Refunded
11        1         999           0.0           Paid


In [3]:
#so fixing fields to calculate net sales 
kpi_df=df.copy()
kpi_df['quantity']=pd.to_numeric(
    kpi_df['quantity'],errors="coerce"
)
kpi_df['discount_pct']=kpi_df['discount_pct'].fillna(0)
kpi_df['payment_status']=kpi_df['payment_status'].str.strip().str.title()
print(kpi_df)

   order_id  order_date customer_segment       city        category  quantity  \
0   RT-1001  2026-01-03          Student    Chennai    Learning Kit       2.0   
1   RT-1002  03/01/2026          Fresher  Bengaluru   Course Access       1.0   
2   RT-1003  2026-01-05          student    Chennai   Course Access       1.0   
3   RT-1004  2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
4   RT-1004  2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
5   RT-1005  2026-01-09          Fresher        NaN  Mentor Session       1.0   
6   RT-1006  2026-13-10          Student       Pune    Learning Kit      -1.0   
7   RT-1007  2026-01-12     Professional     Mumbai  Mentor Session       2.0   
8   RT-1008  2026-01-14          Fresher  Bengaluru   Course Access       NaN   
9   RT-1009  2026-01-16          Student    Chennai    Learning Kit       1.0   
10  RT-1010  2026-01-18     Professional      Delhi   Course Access       2.0   
11  RT-1011         NaN     

In [4]:
# identifying valid records for net sales calculation 
kpi_df["valid_records"]=(
    (kpi_df["order_id"].notna()) & 
    (kpi_df["quantity"]>0) & 
    (kpi_df["unit_price"]>=0) & 
    (kpi_df["discount_pct"].between(0,100)) & 
    (kpi_df["payment_status"]=="Paid")
)
print("Valid records net sales count is ",kpi_df["valid_records"].sum())
print("Invalid records for net sales count is ",(~kpi_df["valid_records"]).sum())

Valid records net sales count is  6
Invalid records for net sales count is  6


In [5]:
print(kpi_df.loc[
    ~kpi_df["valid_records"],["order_id","quantity","unit_price","discount_pct","payment_status"]
    ])

   order_id  quantity  unit_price  discount_pct payment_status
2   RT-1003       1.0        1499           0.0        Pending
5   RT-1005       1.0         999           0.0         Failed
6   RT-1006      -1.0         799          10.0           Paid
7   RT-1007       2.0         999         105.0           Paid
8   RT-1008       NaN        1499           0.0        Pending
10  RT-1010       2.0        1499          15.0       Refunded


# Cleaning data

In [6]:
#checking duplicates 
duplicate_orders=kpi_df[kpi_df['order_id'].duplicated(keep=False)]
display(duplicate_orders)

kpi_df=kpi_df.drop_duplicates()
print("After removing duplicates length of dataset is ",len(kpi_df))

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,valid_records
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3.0,799,5.0,Paid,True
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3.0,799,5.0,Paid,True


After removing duplicates length of dataset is  11


In [7]:
#checking invalid quantity records 
print(kpi_df.loc[
    ((kpi_df["quantity"].isna()) | (kpi_df["quantity"]<=0)) ,["order_id","quantity","category","unit_price"] 
])

  order_id  quantity       category  unit_price
6  RT-1006      -1.0   Learning Kit         799
8  RT-1008       NaN  Course Access        1499


In [8]:
kpi_df.loc[kpi_df['quantity'].isna(),['quantity']]=2
print(kpi_df.loc[
    kpi_df["order_id"].isin(["RT-1006", "RT-1008"]),
    ["order_id", "quantity"]
])

  order_id  quantity
6  RT-1006      -1.0
8  RT-1008       2.0


In [9]:
#removing negative quantity records 
kpi_df=kpi_df[kpi_df['quantity']>0].copy().reset_index(drop=True)
print(kpi_df)

  order_id  order_date customer_segment       city        category  quantity  \
0  RT-1001  2026-01-03          Student    Chennai    Learning Kit       2.0   
1  RT-1002  03/01/2026          Fresher  Bengaluru   Course Access       1.0   
2  RT-1003  2026-01-05          student    Chennai   Course Access       1.0   
3  RT-1004  2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
4  RT-1005  2026-01-09          Fresher        NaN  Mentor Session       1.0   
5  RT-1007  2026-01-12     Professional     Mumbai  Mentor Session       2.0   
6  RT-1008  2026-01-14          Fresher  Bengaluru   Course Access       2.0   
7  RT-1009  2026-01-16          Student    Chennai    Learning Kit       1.0   
8  RT-1010  2026-01-18     Professional      Delhi   Course Access       2.0   
9  RT-1011         NaN          Student      Kochi  Mentor Session       1.0   

   unit_price  discount_pct payment_status  valid_records  
0         799          10.0           Paid           True  

In [10]:
#cleaning discount_pct 
kpi_df['discount_pct']=kpi_df['discount_pct'].fillna(0)
kpi_df=kpi_df[kpi_df['discount_pct'].between(0,100)].copy().reset_index(drop=True)
print(kpi_df)

  order_id  order_date customer_segment       city        category  quantity  \
0  RT-1001  2026-01-03          Student    Chennai    Learning Kit       2.0   
1  RT-1002  03/01/2026          Fresher  Bengaluru   Course Access       1.0   
2  RT-1003  2026-01-05          student    Chennai   Course Access       1.0   
3  RT-1004  2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
4  RT-1005  2026-01-09          Fresher        NaN  Mentor Session       1.0   
5  RT-1008  2026-01-14          Fresher  Bengaluru   Course Access       2.0   
6  RT-1009  2026-01-16          Student    Chennai    Learning Kit       1.0   
7  RT-1010  2026-01-18     Professional      Delhi   Course Access       2.0   
8  RT-1011         NaN          Student      Kochi  Mentor Session       1.0   

   unit_price  discount_pct payment_status  valid_records  
0         799          10.0           Paid           True  
1        1499           0.0           Paid           True  
2        1499      

In [11]:
#Normalizing categorical fields 
kpi_df['customer_segment']=kpi_df['customer_segment'].str.strip().str.title()
kpi_df['payment_status']=kpi_df['payment_status'].str.strip().str.title()
print(kpi_df) 

  order_id  order_date customer_segment       city        category  quantity  \
0  RT-1001  2026-01-03          Student    Chennai    Learning Kit       2.0   
1  RT-1002  03/01/2026          Fresher  Bengaluru   Course Access       1.0   
2  RT-1003  2026-01-05          Student    Chennai   Course Access       1.0   
3  RT-1004  2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
4  RT-1005  2026-01-09          Fresher        NaN  Mentor Session       1.0   
5  RT-1008  2026-01-14          Fresher  Bengaluru   Course Access       2.0   
6  RT-1009  2026-01-16          Student    Chennai    Learning Kit       1.0   
7  RT-1010  2026-01-18     Professional      Delhi   Course Access       2.0   
8  RT-1011         NaN          Student      Kochi  Mentor Session       1.0   

   unit_price  discount_pct payment_status  valid_records  
0         799          10.0           Paid           True  
1        1499           0.0           Paid           True  
2        1499      

In [12]:
#cleaning order_date 
raw_dates=kpi_df['order_date'].astype(str).str.strip()
kpi_df['order_date']=pd.to_datetime(
    raw_dates, format="%Y-%m-%d",errors="coerce"
)
missing_dates=kpi_df['order_date'].isna() & raw_dates.notna()
kpi_df.loc[missing_dates,'order_date']=pd.to_datetime(
    raw_dates[missing_dates],format="%d/%m/%Y",errors="coerce"
)
print(kpi_df)

  order_id order_date customer_segment       city        category  quantity  \
0  RT-1001 2026-01-03          Student    Chennai    Learning Kit       2.0   
1  RT-1002 2026-01-03          Fresher  Bengaluru   Course Access       1.0   
2  RT-1003 2026-01-05          Student    Chennai   Course Access       1.0   
3  RT-1004 2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
4  RT-1005 2026-01-09          Fresher        NaN  Mentor Session       1.0   
5  RT-1008 2026-01-14          Fresher  Bengaluru   Course Access       2.0   
6  RT-1009 2026-01-16          Student    Chennai    Learning Kit       1.0   
7  RT-1010 2026-01-18     Professional      Delhi   Course Access       2.0   
8  RT-1011        NaT          Student      Kochi  Mentor Session       1.0   

   unit_price  discount_pct payment_status  valid_records  
0         799          10.0           Paid           True  
1        1499           0.0           Paid           True  
2        1499           0.0  

In [13]:
kpi_df=kpi_df.dropna(subset=['order_date']).reset_index(drop=True)
print(kpi_df)

  order_id order_date customer_segment       city        category  quantity  \
0  RT-1001 2026-01-03          Student    Chennai    Learning Kit       2.0   
1  RT-1002 2026-01-03          Fresher  Bengaluru   Course Access       1.0   
2  RT-1003 2026-01-05          Student    Chennai   Course Access       1.0   
3  RT-1004 2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
4  RT-1005 2026-01-09          Fresher        NaN  Mentor Session       1.0   
5  RT-1008 2026-01-14          Fresher  Bengaluru   Course Access       2.0   
6  RT-1009 2026-01-16          Student    Chennai    Learning Kit       1.0   
7  RT-1010 2026-01-18     Professional      Delhi   Course Access       2.0   

   unit_price  discount_pct payment_status  valid_records  
0         799          10.0           Paid           True  
1        1499           0.0           Paid           True  
2        1499           0.0        Pending          False  
3         799           5.0           Paid      

In [14]:
kpi_df['city']=kpi_df['city'].fillna("Unknown")
print(kpi_df) 
kpi_df=kpi_df.drop(columns=['valid_records'])
kpi_df.to_csv("../outputs/cleaned_retail_orders.csv",index=False)

  order_id order_date customer_segment       city        category  quantity  \
0  RT-1001 2026-01-03          Student    Chennai    Learning Kit       2.0   
1  RT-1002 2026-01-03          Fresher  Bengaluru   Course Access       1.0   
2  RT-1003 2026-01-05          Student    Chennai   Course Access       1.0   
3  RT-1004 2026-01-07     Professional  Hyderabad    Learning Kit       3.0   
4  RT-1005 2026-01-09          Fresher    Unknown  Mentor Session       1.0   
5  RT-1008 2026-01-14          Fresher  Bengaluru   Course Access       2.0   
6  RT-1009 2026-01-16          Student    Chennai    Learning Kit       1.0   
7  RT-1010 2026-01-18     Professional      Delhi   Course Access       2.0   

   unit_price  discount_pct payment_status  valid_records  
0         799          10.0           Paid           True  
1        1499           0.0           Paid           True  
2        1499           0.0        Pending          False  
3         799           5.0           Paid      